# Slow oscillation and spindle responses to stimulation

This notebook measures how the EEG responds to the sounds played by a Portiloop. For every stimulation, it cuts a
window of signal around the stimulation marker (an **epoch**), discards the epochs contaminated by artifacts, and
averages the others in two frequency bands:

- **Slow band (0.1 to 4 Hz):** slow oscillations; a sound during deep sleep evokes a negative then a positive slow wave.
- **Sigma envelope (11 to 16 Hz):** the amplitude of the spindle band, which rises when the stimulation triggers spindles.

Both bands use the same epochs, whether the Portiloop targeted slow oscillations or spindles.

**How to use it:** edit the configuration cell, then run all the cells from the top (*Run > Run All Cells*) after
every change. With an empty list of files, the notebook writes and analyses a synthetic demo recording.

## 1. Configuration

In [ ]:
from pathlib import Path

# Recording(s) to analyse: the .csv file(s) of one night. Several files of the same night are put in order using
# the date and time in their names, and the time between them is kept as missing data.
# Leave the list empty to try the notebook on a synthetic demo recording.
files = [
    # "/path/to/Par1_2026-10-15_13-55-36_CONDB.csv",
]

channel = "Cz"          # electrode to analyse
montage = "default"     # "default" (ECG1, ECG2, EOG1, EOG2, EMG, Cz, Fz, Fpz) or "no_ecg" (Fpz, Cz, Pz, Fz, EOG1, EOG2, EMG1, EMG2)
notch_hz = 60           # mains frequency to remove: 60 Hz in North America, 50 Hz in Europe
window_s = 5            # epochs from -window_s to +window_s seconds around each stimulation
max_ptp_uv = 350        # reject epochs whose broadband (0.1 to 20 Hz) peak-to-peak amplitude exceeds this (uV)
jump_limit_uv = 1000    # reject epochs with a jump of the raw signal larger than this between two samples (uV); None: no check
sigma_baseline = (-1.5, -1.0)  # window (s) whose mean is subtracted from the sigma envelope; None: absolute envelope

output_name = "slow_and_sigma"  # results go to a folder <first file name>_<output_name> next to the first file

## 2. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from portiloop_analysis import (EpochSettings, event_locked_responses, load_recording, make_demo_recording,
                                plot_responses, read_metadata)

## 3. Load the recording

`load_recording` reads the selected channel and the stimulation markers. It also reads the `_leadoff.json` file
written by the Portiloop: data recorded while the electrode, the reference or the bias was off are treated as
missing.

In [ ]:
if not files:
    files = [make_demo_recording("demo_data", duration_min=30)]
    print("No file given: using a synthetic demo recording.\n")

recording = load_recording(files, channel=channel, montage=montage)
print()
print(recording.summary().to_string())

# Compare the configuration with the settings stored in the recording.
metadata = read_metadata(files[0])
power_line = metadata.get("filter_settings", {}).get("power_line")
if power_line is not None and power_line != notch_hz:
    print(f"\nWarning: the recording was made with power_line = {power_line} Hz, but notch_hz = {notch_hz} Hz.")

## 4. Epochs around the stimulations

The signal is filtered in three bands (each uninterrupted stretch separately, without delay, discarding 10 s at
both ends). An epoch is rejected when it:

| status | reason |
|---|---|
| `edge` | reaches outside the recording |
| `leadoff` | overlaps a period with the electrode, the reference or the bias off |
| `jump` | contains an abrupt change of the raw signal (larger than `jump_limit_uv`) |
| `missing` | contains missing data, or is too close to a gap or to the start or end of a file |
| `artifact` | has a broadband peak-to-peak amplitude larger than `max_ptp_uv` |

The mean of each slow-band epoch is subtracted; the sigma envelope is computed with the Hilbert transform and its
mean in the baseline window is subtracted.

In [ ]:
settings = EpochSettings(window_s=window_s, notch_hz=notch_hz, max_ptp_uv=max_ptp_uv,
                         jump_limit_uv=jump_limit_uv, slow_baseline="epoch", sigma_baseline=sigma_baseline)
responses = event_locked_responses(recording, settings)

print(f"Kept {responses.n_kept} of {len(responses.events)} epochs.")
responses.rejection_summary().to_frame()

## 5. Average responses

Black: average across the kept epochs; grey: +/- 1 standard error of the mean (SEM). Epochs closer than their
length share samples, so the SEM may underestimate the uncertainty.

In [ ]:
fig = plot_responses(responses)
plt.show()

## 6. Save the results

Files written to the output folder:

- `slow_and_sigma_average.png` and `.pdf`: the figure;
- `average_responses.csv`: time, mean and SEM of every band;
- `events.csv`: every stimulation marker (sample, time) and its status (`kept` or the reason of rejection);
- `recording_info.csv`: the files analysed and their position on the timeline.

In [ ]:
output_dir = Path(files[0]).with_name(f"{Path(files[0]).stem}_{output_name}")
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / "slow_and_sigma_average.png", dpi=200, bbox_inches="tight")
fig.savefig(output_dir / "slow_and_sigma_average.pdf", bbox_inches="tight")
responses.averages().to_csv(output_dir / "average_responses.csv", index=False)
responses.events.to_csv(output_dir / "events.csv", index=False)
recording.files.to_csv(output_dir / "recording_info.csv", index=False)
print(f"Saved the results of {responses.n_kept} epochs in {output_dir}")